# Creating arrays

## Learning objectives

By the end of this notebook you will be able to:

- create arrays from lists and nested lists
- use zeros, ones, full, arange, linspace and eye
- generate random numbers reproducibly
- simulate laboratory measurements from known distributions
- load numeric data from a text file

## Introduction

Arrays usually come from data files, but you constantly need to create them yourself too: a range of
doses, a grid of zeros to fill in, or simulated measurements to test an analysis on.

In [1]:
import numpy as np

## From lists

In [2]:
one_d = np.array([12.1, 10.4, 13.8])
two_d = np.array([[1, 2, 3], [4, 5, 6]])
print(one_d.shape, two_d.shape)
print(two_d)

(3,) (2, 3)
[[1 2 3]
 [4 5 6]]


## Filled arrays

In [3]:
print(np.zeros(4))
print(np.ones((2, 3)))
print(np.full((2, 2), 7.5))
print(np.eye(3))                     # identity matrix
print(np.zeros_like(two_d))          # same shape as another array

[0. 0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[7.5 7.5]
 [7.5 7.5]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[0 0 0]
 [0 0 0]]


## Ranges

- `np.arange(start, stop, step)`: like `range`, stop excluded; works with decimals
- `np.linspace(start, stop, num)`: `num` evenly spaced values, **stop included**

In [4]:
print(np.arange(0, 10, 2))
print(np.arange(0, 1, 0.25))
print(np.linspace(0, 1, 5))
doses = np.linspace(0, 100, 11)
print(doses)

[0 2 4 6 8]
[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
[  0.  10.  20.  30.  40.  50.  60.  70.  80.  90. 100.]


For decimal steps, prefer `linspace`: with `arange`, rounding can make the last value appear or
disappear unexpectedly.

A dilution series uses `geomspace` (equal ratios instead of equal steps):

In [5]:
print(np.geomspace(1, 1 / 64, 7))      # two-fold dilutions
print(np.logspace(0, 3, 4))            # 10**0 to 10**3

[1.       0.5      0.25     0.125    0.0625   0.03125  0.015625]
[   1.   10.  100. 1000.]


## Random numbers

`np.random.default_rng(seed)` creates a random number generator. A fixed **seed** gives the same
numbers every time, so your analysis can be repeated exactly.

In [6]:
rng = np.random.default_rng(42)
print(rng.random(3))                       # uniform between 0 and 1
print(rng.integers(1, 7, size=10))         # dice rolls (7 excluded)
print(rng.choice(["A", "B", "C"], size=5))

[0.77395605 0.43887844 0.85859792]
[1 5 2 1 4 6 5 5 5 5]
['B' 'A' 'C' 'B' 'B']


### Distributions for simulation

| function | models | parameters |
|---|---|---|
| `rng.normal(mean, sd, n)` | symmetric measurements (height, haemoglobin) | mean, standard deviation |
| `rng.lognormal(mu, sigma, n)` | positive, right-skewed values (concentrations, viral loads) | on the log scale |
| `rng.binomial(n, p, size)` | number of successes out of n trials | trials, probability |
| `rng.poisson(lam, size)` | counts per unit (colonies per plate, cases per week) | mean count |
| `rng.uniform(low, high, n)` | equally likely values in a range | bounds |

In [7]:
rng = np.random.default_rng(1)
haemoglobin = rng.normal(loc=13.5, scale=1.6, size=200)
print(haemoglobin[:5].round(1))
print(f"mean {haemoglobin.mean():.2f}, sd {haemoglobin.std(ddof=1):.2f}")

[14.1 14.8 14.  11.4 14.9]
mean 13.38, sd 1.48


The sample mean and standard deviation are close to 13.5 and 1.6, but not exact: that is sampling
variation. Larger samples get closer.

In [8]:
for n in [10, 100, 10_000]:
    s = rng.normal(13.5, 1.6, n)
    print(f"n={n:>6}: mean {s.mean():.3f}")

n=    10: mean 13.787
n=   100: mean 13.224
n= 10000: mean 13.489


In [9]:
colonies = rng.poisson(lam=35, size=12)              # colonies on 12 plates
positives = rng.binomial(n=50, p=0.12, size=8)       # positives per batch of 50 tests
viral_load = rng.lognormal(mean=8, sigma=1.5, size=6)
print(colonies)
print(positives)
print(viral_load.round(0))

[36 41 34 24 44 45 33 28 30 30 29 36]
[7 4 9 3 8 7 9 3]
[17976.  3100.   101. 87533.   428.  1326.]


## A simulated experiment

Three replicate measurements for each of 5 samples, as a 5 × 3 array:

In [10]:
rng = np.random.default_rng(7)
true_values = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
replicates = true_values[:, None] + rng.normal(0, 0.3, size=(5, 3))
print(replicates.round(2))
print(replicates.shape)

[[ 2.    2.09  1.92]
 [ 3.73  3.86  3.7 ]
 [ 6.02  6.4   5.85]
 [ 7.81  8.15  8.11]
 [10.03  9.72  9.99]]
(5, 3)


`true_values[:, None]` turns the 5 values into a column so they can be added to a 5 × 3 grid; this is
*broadcasting*, explained in notebook 04.

## Loading numbers from a text file

`np.loadtxt` and `np.genfromtxt` read numeric text files. `genfromtxt` copes with missing values.
pandas is usually easier for real tables (module 6).

In [11]:
expr = np.genfromtxt("../datasets/raw/gene_expression.tsv", delimiter="\t", skip_header=1,
                     usecols=range(2, 22))
print(expr.shape, expr.dtype)
print(expr[:3, :5].round(2))

(24, 20) float64
[[ 8.03  7.85  8.4   7.99  7.13]
 [ 9.32  8.26  7.92 10.02  7.85]
 [ 7.27  9.9   8.57  7.26  6.09]]


## Saving and loading arrays

In [12]:
np.save("replicates.npy", replicates)
back = np.load("replicates.npy")
print(np.array_equal(back, replicates))

np.savetxt("replicates.csv", replicates, delimiter=",", fmt="%.3f")
print(open("replicates.csv").read())

import os
os.remove("replicates.npy"); os.remove("replicates.csv")

True
2.000,2.090,1.918
3.733,3.864,3.703
6.018,6.402,5.852
7.814,8.147,8.107
10.032,9.721,9.991



## Practise

Exercises for this folder are in `exercises/NumPy_Exercises.ipynb`.

## Further reading

- [NumPy user guide](https://numpy.org/doc/stable/user/index.html)
- [Array creation](https://numpy.org/doc/stable/user/basics.creation.html)
- [Random generator](https://numpy.org/doc/stable/reference/random/generator.html)